In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

# ============ Config ============
experiment_name = "baseline"
log_dir = f"runs/{experiment_name}"
save_path = f"runs/{experiment_name}/best.pth"
num_epochs = 40
batch_size = 64
learning_rate = 0.001
# ================================

# 1. Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. TensorBoard SummaryWriter
writer = SummaryWriter(log_dir)

# 3. Transform (no augmentation)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 4. Dataset & DataLoader
train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

# 5. Model (fix warning: use weights=None)
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 10)
model.to(device)

# 6. Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 7. Evaluation function
def evaluate():
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

# 8. Training function
def train_one_epoch(epoch):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

    avg_loss = running_loss / len(train_loader)
    train_acc = correct / total
    val_acc = evaluate()

    # Log to TensorBoard
    writer.add_scalar("Loss/train", avg_loss, epoch)
    writer.add_scalar("Accuracy/train", train_acc, epoch)
    writer.add_scalar("Accuracy/val", val_acc, epoch)

    print(f"Epoch [{epoch+1}] Train Loss: {avg_loss:.4f}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")
    return val_acc

# 9. Training loop with best model saving
best_val_acc = 0.0
for epoch in range(num_epochs):
    val_acc = train_one_epoch(epoch)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), save_path)
        print(f"✅ New best model saved with val acc: {best_val_acc:.4f}")

writer.close()

Epoch [1] Train Loss: 1.3600, Train Acc: 0.5109, Val Acc: 0.6007
✅ New best model saved with val acc: 0.6007
Epoch [2] Train Loss: 0.9713, Train Acc: 0.6600, Val Acc: 0.6402
✅ New best model saved with val acc: 0.6402
Epoch [3] Train Loss: 0.7993, Train Acc: 0.7200, Val Acc: 0.7080
✅ New best model saved with val acc: 0.7080
Epoch [4] Train Loss: 0.6761, Train Acc: 0.7647, Val Acc: 0.7242
✅ New best model saved with val acc: 0.7242
Epoch [5] Train Loss: 0.5755, Train Acc: 0.8011, Val Acc: 0.7636
✅ New best model saved with val acc: 0.7636
Epoch [6] Train Loss: 0.4806, Train Acc: 0.8335, Val Acc: 0.7516
Epoch [7] Train Loss: 0.3974, Train Acc: 0.8644, Val Acc: 0.7639
✅ New best model saved with val acc: 0.7639
Epoch [8] Train Loss: 0.3200, Train Acc: 0.8884, Val Acc: 0.7570
Epoch [9] Train Loss: 0.2635, Train Acc: 0.9081, Val Acc: 0.7655
✅ New best model saved with val acc: 0.7655
Epoch [10] Train Loss: 0.2047, Train Acc: 0.9290, Val Acc: 0.7611
Epoch [11] Train Loss: 0.1699, Train Acc:

In [1]:
import subprocess
import time
import webbrowser

def start_tensorboard(logdir="runs", port=6006):
    """
    Starts TensorBoard as a background process and opens it in your default browser.
    Returns the process handle so you can terminate it later.
    """
    tb_cmd = [
        "tensorboard",
        f"--logdir={logdir}",
        f"--port={port}",
        "--host=localhost"
    ]
    
    # Start the process
    print(f"Starting TensorBoard on port {port}...")
    process = subprocess.Popen(tb_cmd)
    
    # Give it a second to get going
    time.sleep(2)
    
    # Open in browser automatically
    url = f"http://localhost:{port}"
    print(f"Opening {url} in your default browser...")
    webbrowser.open(url)
    
    return process


def stop_tensorboard(process):
    """
    Stops the TensorBoard process started by `start_tensorboard()`.
    """
    print("Stopping TensorBoard...")
    process.terminate()  # or process.kill()

In [2]:
process = start_tensorboard(logdir="runs", port=6012)

Starting TensorBoard on port 6012...


TensorFlow installation not found - running with reduced feature set.

NOTE: Using experimental fast data loading logic. To disable, pass
    "--load_fast=false" and report issues on GitHub. More details:
    https://github.com/tensorflow/tensorboard/issues/4784

TensorBoard 2.19.0 at http://localhost:6012/ (Press CTRL+C to quit)


Opening http://localhost:6012 in your default browser...


In [4]:
# stop_tensorboard(process)
# Uncomment the line below to stop TensorBoard when you're done